<a href="https://colab.research.google.com/github/Rohit-Roby/Project/blob/main/SSID_2235053.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install objaverse
!pip install trimesh
!pip install open3d
!pip install torchsummary

In [ ]:
# Import libraries
import objaverse
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string
import trimesh
import os
import shutil
import multiprocessing
nltk.download('stopwords')
import numpy as np
from matplotlib import pyplot as plt
import open3d as o3d
import torch
import torchsummary

**Loading the dataset files and dowloading 3d model**

In [ ]:
#loading Uids
uids = objaverse.load_uids()
len(uids), type(uids)


In [ ]:
len(uids)

In [ ]:

# lvis_annotations
lvis_annotations = objaverse.load_lvis_annotations()

# Sort the dictionary by value in descending order
items = dict(sorted(lvis_annotations.items(), key=lambda item: len(item[1]), reverse=True))
items.keys()

In [ ]:
top_items = list(items)
chairs = top_items[0]

In [ ]:
chairs = items[chairs]

In [ ]:
chairs

In [ ]:
len(items)

In [ ]:

# Plot the keys with values
plt.xticks(rotation='vertical')
plt.plot(list(items.keys()), [len(value) for value in items.values()])
plt.show()


In [ ]:
uids = []  # Initialize an empty list
for uid_list in items.values():
    uids.extend(uid_list)  # Add UIDs individually


In [ ]:
len(uids)

In [ ]:
t_uids = uids[:5000]

In [ ]:
len(t_uids)

In [ ]:
# annotations = objaverse.load_annotations(t_uids)
annotations = objaverse.load_annotations(chairs)


In [ ]:
import pandas as pd
df = pd.DataFrame()
df['uids'] = t_uids



**Preprocessing of Text Description**

In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
def clean_description(description):
    # Define stop words
    stop_words = set(stopwords.words("english"))


    # Remove punctuation
    text = description.translate(str.maketrans('', '', string.punctuation))

    # Remove stop words
    text = ' '.join([word for word in text.split() if word not in stop_words])

    # Lowercase conversion
    text = text.lower()

    # Stemming (or lemmatization)
    stemmer = PorterStemmer()
    text = ' '.join([stemmer.stem(word) for word in text.split()])

    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

    return text


# df['description'] = df['uids'].map(lambda uid: clean_description( annotations[uid]['description']))

In [ ]:
# objaverse_description = []
for annotation in annotations.values():
    if isinstance(annotation, dict):
        tags = annotation.get("tags")  # returns 'None' if key is missing
        if tags:  # Check if tags exists
            tag_descriptions = [tag['name'] for tag in tags]
            description = " ".join(tag_descriptions)
             # Store tags and description with annotation
            annotation["description"] = clean_description(description)  # adding a description field
            # objaverse_description.append(description)

In [ ]:
import numpy as np
df['description'].replace('', np.nan, inplace=True)
df = df.dropna()

In [ ]:
processes = multiprocessing.cpu_count()
processes
objects = objaverse.load_objects(
    uids = t_uids,
    download_processes = processes
)

In [ ]:
# Define the subfolder and parent folder paths
parent_dir = '/root/.objaverse/hf-objaverse-v1/glbs'

# Walk through the parent directory
for subdir, dirs, files in os.walk(parent_dir):
    for file in files:

        file_path = os.path.join(subdir, file)
        dest_path = os.path.join(parent_dir, file)
        # Move the file to the parent directory
        shutil.move(file_path, dest_path)
    if subdir != parent_dir:
        os.rmdir(subdir)

df['glb_path'] = df['uids'].map(lambda x: objects[x])

In [ ]:
df.head()

In [ ]:
import re
# Function to remove "/glbs"
def remove_glbs(path):
    return re.sub(r'\d{3}-\d{3}/', '', path)    # Replace '/glbs' with an empty string

# Apply the function using map
df['glb_path'] = df['glb_path'].map(remove_glbs)

In [ ]:
import time

def verbose_time(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"{func.__name__} took {end_time - start_time:.2f} seconds to execute.")
        return result
    return wrapper

**Preprocessing of 3d model into point cloud**

In [ ]:
import os
def get_model_paths(base_dir):
    model_paths = []
    for subdir, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith(".glb"):
                file_path = os.path.join(subdir, file)
                model_paths.append(file_path)
    return model_paths
base_dir ='/root/.objaverse/hf-objaverse-v1/glbs'
model_paths = get_model_paths(base_dir)

In [ ]:
df.head()

In [ ]:
# df.to_csv('/root/.objaverse/Dataset')
df.to_csv('/content/drive/MyDrive/DATA/dataset.csv')

import pandas as pd
# df = pd.read_csv('/root/.objaverse/Dataset')
df = pd.read_csv('/content/drive/MyDrive/DATA/dataset.csv')

In [ ]:
glbs = model_paths[0:3]

for glb in glbs:
  if glb:
      mesh = o3d.io.read_triangle_mesh(glb)
      o3d.visualization.draw_plotly([mesh])


**voxelization using open3d**

In [ ]:
dir = '/root/.objaverse/hf-objaverse-v1/glbs'


In [ ]:
df.head()

In [ ]:
dataSet = {}

In [ ]:
glb = model_paths[0]
file, _ = os.path.split(glb)[1].split('.glb')

In [ ]:
file

In [ ]:
@verbose_time
def o3d_voxelize(filepath):
    mesh = o3d.io.read_triangle_mesh(filepath)
    # os.path.split(filepath)[]
    print(mesh)
    id, _ = os.path.split(filepath)[1].split('.glb')
    # mesh.compute_vertex_normals()
    # fit to unit cube
    mesh.scale(1 / np.max(mesh.get_max_bound() - mesh.get_min_bound()),center=mesh.get_center())
    # o3d.visualization.draw([mesh])
    # print('voxelization')
    voxel_grid = o3d.geometry.VoxelGrid.create_from_triangle_mesh(mesh, voxel_size=0.05)
    # o3d.visualization.draw([voxel_grid])
    dataSet[id] = voxel_grid
    return voxel_grid

In [ ]:
o3d_voxelize()

In [ ]:
@verbose_time
#convert to pytorch tensors
def voxel_grid_to_tensor(voxel_grid, target_shape):

    voxels = np.asarray(voxel_grid.get_voxels())
    # if len(voxels) == 0:
    #       # Return an empty tensor if the voxel grid is empty
    #     return torch.zeros(target_shape, dtype=torch.float32)
    voxel_positions = np.array([voxel.grid_index for voxel in voxels])
    if voxel_positions.size == 0:
       return torch.zeros(target_shape, dtype=torch.float32)  # Return an empty tensor if no non-empty voxels


    max_coords = voxel_positions.max(axis=0)
    tensor_shape = tuple(max_coords + 1)
    voxel_tensor = torch.zeros(tensor_shape, dtype=torch.float32)
    voxel_tensor[tuple(voxel_positions.T)] = 1

    # Rescale tensor to target shape
    scaling_factors = [target_shape[i] / voxel_tensor.shape[i] for i in range(3)]
    # Use nearest neighbor scaling
    voxel_tensor_rescaled = torch.tensor(zoom(voxel_tensor, scaling_factors, order=0))

    # Print the final shape of the tensor
    # print("Final tensor shape:", voxel_tensor_rescaled.shape)

    return voxel_tensor_rescaled



In [ ]:
voxel_grid_to_tensor()

In [ ]:
df.head()

**PRETRAINED TEXT ENCODER USING T5**

In [ ]:
import torch
import torch.nn as nn
import transformers
from transformers import T5ForConditionalGeneration, AutoTokenizer

In [ ]:

class TextEncoder(nn.Module):
    def __init__(self, model_name="t5-large"):  # T5 model size
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.transformer = T5ForConditionalGeneration.from_pretrained(model_name)

    def forward(self, text_input):
        input = self.tokenizer(text_input, return_tensors="pt")  # Tokenize input
        start_token_id = self.tokenizer.unk_token_id   # start token for your model.
        decoder_input_ids = torch.tensor([[start_token_id]])
        input['decoder_input_ids'] = decoder_input_ids
        # print(input)
        outputs = self.transformer(**input)  # T5 model

        # hidden_states = outputs.last_hidden_state  # Option 1: Last hidden state
        hidden_states = outputs.encoder_last_hidden_state

        # Simple mean pooling
        latent_representation = hidden_states.mean(dim=1)

        return latent_representation

In [ ]:
sample_descriptions = ["A modern living room A minimalist bedroom A chair "]
text_encoder = TextEncoder()
for description in sample_descriptions:
    latent_representation = text_encoder(description)
    print(f"Description: {description}")
    print(f"Latent Representation: {latent_representation.shape}") # Check the shape
    print("--------------")

**GENERATOR AND DISCRIMINATOR MODELS**

In [ ]:
import torch.nn as nn

# Generator architecture
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super(Generator, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 512 * 4 * 4 * 4),
            nn.ReLU(True)
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose3d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(True),
            nn.ConvTranspose3d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(True),
            nn.ConvTranspose3d(128, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.fc(x)
        x = x.view(-1, 512, 4, 4, 4)  # Reshape to 3D volume
        x = self.deconv(x)
        return x


In [ ]:

# Discriminator architecture
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout3d(0.1)
        )
        self.fc = nn.Sequential(
            nn.Linear(256 * 4 * 4 *4 , 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(-1, 256 * 4 * 4 *4 )  # Flatten
        x = self.fc(x)
        return x


**Model Training**

In [ ]:

text_descriptions = df['description'].tolist()


In [ ]:
# type(voxel_tensors)
from scipy.ndimage import zoom

voxel_grids = [o3d_voxelize(filepath) for filepath in model_paths]

target_shape = (32,32,32)
for id in dataSet.keys():
    dataSet[id] = voxel_grid_to_tensor(dataSet[id], target_shape)



In [ ]:

for id, tensor in dataSet.items():
    dataSet[id] = tensor.unsqueeze(0)

In [ ]:
# voxel_tensors_sqe[19].shape

In [ ]:
# tensor_shapes = []

# for tensor in dataSet.values():
#     tensor_shapes.append(tensor.shape)

In [ ]:
updated_dataSet = {}

for key, tensor in dataSet.items():
    if tensor.numel() > 0:  # Check if tensor has values
        updated_dataSet[key] = tensor
    elif tensor.shape != torch.Size([]):  # Check if tensor has a non-empty shape
        updated_dataSet[key] = tensor

dataSet = updated_dataSet

In [ ]:
# dataSet.values())

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
# voxel_tensors_sq = [tensor.to(device) for tensor in voxel_tensors_sq]
for id, tensor in dataSet.items():
    dataSet[id] = tensor.to(device)

In [ ]:
import numpy as np
# np.save('/content/drive/MyDrive/DATA/chair_dataSet.npy',dataSet)

In [ ]:
dataSet = np.load('/content/drive/MyDrive/DATA/chair_dataSet.npy', allow_pickle=True).item()

In [ ]:
ndataSet = np.load('/content/dataSet.npy', allow_pickle=True).item()

In [ ]:
dataSet['0017fed1dd804ec1ab2e06bbdbebab16'].squeeze(1).shape

In [ ]:
gen_path = 'generator.pth'
dis_path = 'discriminator.pth'

In [ ]:
# generator.load_state_dict(torch.load(gen_path))
# discriminator.load_state_dict(torch.load(dis_path))

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
    def __init__(self, dataSet):
        self.dataSet = dataSet
        self.ids = list(dataSet.keys())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id = self.ids[idx]
        voxel_tensor = self.dataSet[id]
        return voxel_tensor, id

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

batch_size = 10
num_epochs = 100
learning_rate = 0.001
custom_dataset = CustomDataset(dataSet)
train_loader = torch.utils.data.DataLoader(
     custom_dataset, batch_size=batch_size, shuffle=True
)
noise_dim = 1024
# latent_dim = latent_dim.astype(torch.float32)
#define generator and discriminator networks
generator = Generator(noise_dim)
discriminator = Discriminator()
generator = generator.to(device)
discriminator = discriminator.to(device)

def weights_init(m):
    if isinstance(m, nn.Conv3d) or isinstance(m, nn.ConvTranspose3d):
        nn.init.kaiming_normal_(m.weight.data, nonlinearity='relu')
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight.data)

discriminator_losses =[]
generator_losses =[]
# initialization of weights to the networks
generator.apply(weights_init)
discriminator.apply(weights_init)

# optimizers
criterion = nn.BCELoss().to(device)
optimizer_discriminator = torch.optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_generator = torch.optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))

def weights_init(m):
    if isinstance(m, nn.Conv3d) or isinstance(m, nn.ConvTranspose3d):
        nn.init.kaiming_normal_(m.weight.data, nonlinearity='relu')
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight.data)

def train_gen(train_loader):
    for n, (voxelgrid_tensor, id) in enumerate(train_loader):
        optimizer_generator.zero_grad()
        optimizer_discriminator.zero_grad()


        latent_representation =[]
        # prompts = []
        # for i in id:
        #     prompts.append(annotations[i]['description'])

        voxel_grid_tensor = voxelgrid_tensor.to(device)
        curr_batch_size = voxel_grid_tensor.size(0)

        # real_labels = torch.ones(curr_batch_size, 1).to(device)
        # fake_labels = torch.zeros(curr_batch_size, 1).to(device)
        real_labels = torch.ones((curr_batch_size, 1)).to(device)
        fake_labels = torch.zeros(curr_batch_size, 1).to(device)

        # fake_labels = torch.zeros(curr_batch_size, 1).to(device)
        # latent_space_samples = torch.randn((curr_batch_size, latent_dim)).to(device)
        # noise = torch.randn((curr_batch_size, noise_dim)).to(device)
        # latent_space_samples = torch.randn((batch_size, latent_dim)).to(device)
        noise = torch.randn((curr_batch_size, noise_dim)).to(device)
        # print('latent_dim: ', latent_dim)
        # print('noise_dim: ', noise_dim)
        # for text in prompts:
        #   latent_representation.append(text_encoder(text))
        # print('latent_representation :', latent_representation[0].shape)

        # Obtain latent descriptions from text encoder
        # latent_descriptions = text_encoder(prompts)  # Adjust according to your text encoder
        # print('laten_space_sample :', latent_space_samples.shape, 'noise :', noise.shape)

        generated_samples = generator(noise)

        # Training the discriminator

        # for i,latent_description in enumerate(latent_representation):
            # Ensure voxel_features is 5D
            # if len(voxel_grid_tensor.shape) < 5:
            #     voxel_grid_tensor = voxel_grid_tensor.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
            # voxel_tensor_single = voxel_grid_tensor[i].unsqueeze(0)  # Add a batch dimension
            # generated_sample_single = generated_samples[i].unsqueeze(0)

            # print('voxel_grid_tensor.detach()',voxel_grid_tensor.detach().shape)
            # print('generated_samples.detach()', generated_samples.detach().shape)
            # print('latent_description.to(device)', latent_description.to(device).shape)

        output_discriminator_real = discriminator(voxel_grid_tensor).to(device)
        output_discriminator_fake = discriminator(generated_samples).to(device)

        # print(' output_discriminator_real',  output_discriminator_real.shape)
        # print(' output_discriminator_fake',  output_discriminator_fake.shape)
        # print('real_labels.shape',real_labels.shape)
        # print('fake_labels.shape',fake_labels.shape)


        loss_discriminator_real = criterion(output_discriminator_real, real_labels)
        loss_discriminator_fake = criterion(output_discriminator_fake, fake_labels)
        discriminator_loss = (loss_discriminator_real + loss_discriminator_fake) / 2

        discriminator_loss.backward()
        optimizer_discriminator.step()
    # discriminator_loss.backward()


    # Training the generator
        generated_samples = generator(noise).to(device)
        output_discriminator_generated = discriminator(generated_samples)

        generator_loss = criterion(output_discriminator_generated, real_labels)
        generator_loss.backward()
        optimizer_generator.step()

        discriminator_losses.append(discriminator_loss)
        generator_losses.append(generator_loss)

        if (epoch % 10 == 0) and (n == len(train_loader) - 1):
            print(f"Epoch: {epoch} Loss D.: {discriminator_loss.item()}")
            print(f"Epoch: {epoch} Loss G.: {generator_loss.item()}")



for epoch in range(num_epochs):
    # for name, param in discriminator.named_parameters():
        # if param.requires_grad:
            # print(name, param.grad)

    train_gen(train_loader)

In [ ]:
# Plot the loss values
plt.figure(figsize=(10, 5))
plt.title("Generator and Discriminator Loss During Training")
generator_losses = [generator_loss.to('cpu').detach() for generator_loss in generator_losses]
discriminator_losses = [discriminator_loss.to('cpu').detach() for discriminator_loss in discriminator_losses]
plt.plot(generator_losses, label="G")
plt.plot(discriminator_losses, label="D")
plt.xlabel("iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:

# save models
torch.save(generator.state_dict(), 'generator.pth')
torch.save(discriminator.state_dict(), 'discriminator.pth')


In [ ]:
res = generator(torch.randn(noise_dim).to(device))
# middle slice along the first dimension
out = res.cpu()
slice_2d = out[0, 0, 16, :, :].detach().numpy()

# Plot the slice
plt.imshow(slice_2d, cmap='gray')
plt.title('Middle Slice Visualization')
plt.show()

In [ ]:
# !pip install "git+https://github.com/facebookresearch/pytorch3d.git@stable"

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:

class ConditionalGenerator(nn.Module):
    def __init__(self, latent_dim, noise_dim):
        super(ConditionalGenerator, self).__init__()

        self.latent_projection = nn.Linear(latent_dim, 512 * 4 * 4 * 4)

        self.deconv = nn.Sequential(
            nn.ConvTranspose3d(512 + noise_dim, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(True),
            nn.ConvTranspose3d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(True),
            nn.ConvTranspose3d(128, 1, kernel_size=4, stride=2, padding=1),
            # nn.ConvTranspose3d(128, 64, kernel_size=4, stride=2, padding=1),

            # nn.BatchNorm3d(64),
            # nn.ReLU(True),
            # nn.ConvTranspose3d(64, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh()  # Output likely in range [-1, 1]
        )

    def forward(self, latent_description, noise):
        projected_latent = self.latent_projection(latent_description)
        # print('projected_latent :',projected_latent.shape)
        projected_latent = projected_latent.view(-1, 512, 4, 4, 4)
        # print('projected_latent :',projected_latent.shape)
        # print('noise :',noise.shape)
        # Add dimensions
        noise = noise.unsqueeze(2).unsqueeze(3).unsqueeze(4)
        noise = noise.repeat(1, 1, 4, 4, 4)
        # print('noise unsqueeze :',noise.shape)


        x = torch.cat((projected_latent, noise), dim=1)
        x = self.deconv(x)
        return x


In [ ]:
from torchsummary import summary

In [ ]:

latent_dim = 1024
noise_dim = 1024
generator = ConditionalGenerator(latent_dim, noise_dim)

# Print the summary of the model
summary(generator, input_size=[(latent_dim,), (noise_dim,)])


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
class ConditionalDiscriminator(nn.Module):
    def __init__(self, latent_dim):
        super(ConditionalDiscriminator, self).__init__()
        # architecture for processing the voxel tensor
        self.conv = nn.Sequential(
            nn.Conv3d(1, 64, kernel_size=7, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(64, 128, kernel_size=7, stride=2, padding=1),
            nn.BatchNorm3d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(128, 256, kernel_size=7, stride=2, padding=1),
            nn.BatchNorm3d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout3d(0.4)
        )
        # architecture for processing the latent description
        self.fc_latent = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2, inplace=True)
        )
        # Combine the outputs from the voxel tensor and latent description
        self.fc_combined = nn.Sequential(
            nn.Linear(512, 1),
            nn.Sigmoid()
        )

    def forward(self, voxel_tensor, latent_description):
        # print('voxel feature before conv in discriminator:',voxel_tensor.shape)
        voxel_features = self.conv(voxel_tensor)
        # print('voxel feature conv shape in discriminator:',voxel_features.shape)
        # Flatten the voxel features
        voxel_features = voxel_features.view(-1, 256 * 1 * 1 * 1)
        # print('voxel feature shape in discriminator:',voxel_features.shape)
        # Process the latent description
        latent_features = self.fc_latent(latent_description)
        # print('latent_features in discriminator fc_latent', latent_features.shape)
        latent_features = latent_features.expand(voxel_features.size(0), -1,-1)
        latent_features = latent_features.view(-1,256)
        # print('latent_features in discriminator view', latent_features.shape)

        # if voxel_features.size(0) > latent_features.size(0):
        #     latent_features = latent_features.expand(voxel_features.size(0), -1)
        # print('latent_features in discriminator view', latent_features.shape)
        # latent_features = latent_features.repeat(voxel_tensor.size(0), 1, 1, 1, 1)
        # latent_features = latent_features.repeat(32,1,1,1,1)
        # print('latent_features.shape in discriminator',latent_features.shape)
         # Concatenate the features
        combined_features = torch.cat((voxel_features, latent_features), 1)
        # print('combined features', combined_features.shape)
        # combined_features = combined_features.view(-1, 512)
        # print('combined features', combined_features.shape)
        # validity = self.fc_combined(combined_features).view(-1,1)
        validity = self.fc_combined(combined_features)
        valididty = validity.squeeze(-1)
        return validity


In [ ]:
# from torchsummary import summary

# latent_dim = 1024
# discriminator = ConditionalDiscriminator(latent_dim)

# # Create a random input tensor for the voxel tensor
# voxel_tensor = torch.randn(1, 32, 32, 32)  # Example: batch size 1, 32x32x32 voxel tensor
# latent_description = torch.randn(1, latent_dim)  # Example: batch size 1, latent dim 1024

# # Print the summary of the model
# summary(discriminator, input_size=[(1, 32, 32, 32), (1, latent_dim)])



In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
    def __init__(self, dataSet):
        self.dataSet = dataSet
        self.ids = list(dataSet.keys())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id = self.ids[idx]
        latent = text_encoder(annotations[id]['description'])
        voxel_tensor = self.dataSet[id].squeeze(0)
        return voxel_tensor, latent

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

text_encoder = TextEncoder()
batch_size = 128
num_epochs = 50
learning_rate = 0.001
custom_dataset = CustomDataset(dataSet)
train_loader = torch.utils.data.DataLoader(
     custom_dataset, batch_size=batch_size, shuffle=True
)
latent_dim = 1024
noise_dim = 1024
# latent_dim = latent_dim.astype(torch.float32)
#define generator and discriminator networks
generator = ConditionalGenerator(latent_dim, noise_dim)
discriminator = ConditionalDiscriminator(latent_dim)
generator = generator.to(device)
discriminator = discriminator.to(device)

def weights_init(m):
    if isinstance(m, nn.Conv3d) or isinstance(m, nn.ConvTranspose3d):
        nn.init.kaiming_normal_(m.weight.data, nonlinearity='relu')
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight.data)

discriminator_losses =[]
generator_losses =[]
# initialization of weights to the networks
generator.apply(weights_init)
discriminator.apply(weights_init)

# Optimizers
criterion = nn.BCELoss().to(device)
optimizer_discriminator = torch.optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_generator = torch.optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))

discriminator_losses_epoch = []
generator_losses_epoch = []


In [ ]:

for epoch in range(num_epochs):
  print('Epoch: ', epoch)
  for i, (voxelgrid_tensor, latent) in enumerate(train_loader):
      # print('epoch :',epoch,' iteration :',i , voxelgrid_tensor.shape, latent.shape)

      optimizer_generator.zero_grad()
      voxel_grid_tensor = voxelgrid_tensor.to(device)
      latent = latent.to(device)
      curr_batch_size = voxel_grid_tensor.size(0)

      real_labels = torch.ones((curr_batch_size, 1)).to(device)
      fake_labels = torch.zeros(curr_batch_size, 1).to(device)

      latent_space_samples = torch.randn((curr_batch_size, latent_dim)).to(device)
      noise = torch.randn((curr_batch_size, noise_dim)).to(device)
      generated_samples = generator(latent_space_samples, noise).to(device)

      # Training the discriminator
      optimizer_discriminator.zero_grad()
      # print('voxel_grid tensor', voxel_grid_tensor.shape)
      output_discriminator_real = discriminator(voxel_grid_tensor, latent).to(device)
      output_discriminator_fake = discriminator(generated_samples, latent).to(device)

      # print(' output_discriminator_real',  output_discriminator_real.shape)
      # print(' output_discriminator_fake',  output_discriminator_fake.shape)
      # print('real_labels.shape',real_labels.shape)
      # print('fake_labels.shape',fake_labels.shape)

      loss_discriminator_real = criterion(output_discriminator_real, real_labels).to(device)
      loss_discriminator_fake = criterion(output_discriminator_fake, fake_labels).to(device)
      discriminator_loss = (loss_discriminator_real + loss_discriminator_fake) / 2
      print('e -',epoch,'-d-',discriminator_loss.item())
      discriminator_losses.append(discriminator_loss)

      discriminator_loss.backward()
      optimizer_discriminator.step()

  # Training the generator
      optimizer_generator.zero_grad()
      generated_samples = generator(latent_space_samples, noise).to(device)
      output_discriminator_generated = discriminator(generated_samples.detach(), latent.detach()).to(device)

      generator_loss = criterion(output_discriminator_generated, real_labels).to(device)
      print('e -',epoch,'g-',generator_loss.item())
      generator_losses.append(generator_loss)
      generator_loss.backward()
      optimizer_generator.step()

    # if (epoch % 10 == 0) and (i == len(train_loader) - 1):
  print(f"Epoch: {epoch} Loss D.: {discriminator_loss.item()}")
  print(f"Epoch: {epoch} Loss G.: {generator_loss.item()}")




In [ ]:
generator_losses

In [ ]:
discriminator_losses

In [ ]:
import matplotlib.pyplot as plt

#lists of discriminator and generator losses
discriminator_losses = [loss.cpu().detach().numpy() for loss in discriminator_losses]
generator_losses = [loss.cpu().detach().numpy() for loss in generator_losses]
epochs = range(1, len(discriminator_losses) + 1)  # Create list of epochs
print(len(discriminator_losses))
print(len(generator_losses))
# Plot the discriminator and generator losses
plt.plot(epochs, discriminator_losses, label='Discriminator Loss')
plt.plot(epochs, generator_losses, label='Generator Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Discriminator vs. Generator Loss')

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
text = 'chair and dining table'
latent_d = text_encoder(text)
noise = torch.randn(noise_dim).unsqueeze(0).to(device)
noise.shape

In [ ]:
latent_d.shape

In [ ]:
res = generator(latent_d.to(device),noise)
# the middle slice along the first dimension
out = res.cpu()
slice_2d = out[0, 0, 16, :, :].detach().numpy()

# Plot the slice
plt.imshow(slice_2d, cmap='gray')
plt.title('Middle Slice Visualization')
plt.show()

In [ ]:
dmesh = res.squeeze().detach().cpu().numpy()

In [ ]:
import trimesh
import skimage
def voxel_to_mesh(voxel_grid, threshold=0.1, smoothing= False):
  vertices, faces, normals, values = skimage.measure.marching_cubes(voxel_grid, level=threshold)
  mesh = trimesh.Trimesh(vertices=vertices, faces=faces)

  if smoothing:
    mesh.vertices -=mesh.center_mass
    mesh.vertices /=mesh.scale
    mesh.apply_translation(-mesh.center_mass)
    mesh.apply_scale(1.0/mesh.scale)
  return mesh
mesh = voxel_grid = voxel_to_mesh(dmesh)
mesh.show()

In [ ]:
import random
# Define a custom dataset class for NeRF training
class NeRFDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        pos = self.data[idx]

        # Generate random viewing direction (normalized)
        view_dir = torch.tensor([random.uniform(-1, 1), random.uniform(-1, 1), random.uniform(-1, 1)])
        view_dir /= torch.norm(view_dir)

        # Generate random RGB color
        color = torch.tensor([random.uniform(0, 1), random.uniform(0, 1), random.uniform(0, 1)])

        return pos, view_dir, color


In [ ]:
 # Initialize the custom dataset
    nerf_dataset = NeRFDataset(dataset)
    batch_size = 32
    train_loader = DataLoader(nerf_dataset, batch_size=batch_size, shuffle=True)

In [ ]:

def render_rays(nerf_model, ray_origins, ray_directions, hn=0, hf=0.5, nb_bins=192):
    device = ray_origins.device
    t = torch.linspace(hn, hf, nb_bins, device=device).expand(ray_origins.shape[0], nb_bins)
    # Perturb sampling along each ray.
    mid = (t[:, :-1] + t[:, 1:]) / 2.
    lower = torch.cat((t[:, :1], mid), -1)
    upper = torch.cat((mid, t[:, -1:]), -1)
    u = torch.rand(t.shape, device=device)
    t = lower + (upper - lower) * u  # [batch_size, nb_bins]
    delta = torch.cat((t[:, 1:] - t[:, :-1], torch.tensor([1e10], device=device).expand(ray_origins.shape[0], 1)), -1)

    x = ray_origins.unsqueeze(1) + t.unsqueeze(2) * ray_directions.unsqueeze(1)   # [batch_size, nb_bins, 3]
    ray_directions = ray_directions.expand(nb_bins, ray_directions.shape[0], 3).transpose(0, 1)

    colors, sigma = nerf_model(x.reshape(-1, 3), ray_directions.reshape(-1, 3))
    colors = colors.reshape(x.shape)
    sigma = sigma.reshape(x.shape[:-1])

    alpha = 1 - torch.exp(-sigma * delta)  # [batch_size, nb_bins]
    weights = compute_accumulated_transmittance(1 - alpha).unsqueeze(2) * alpha.unsqueeze(2)
    c = (weights * colors).sum(dim=1)  # Pixel values
    weight_sum = weights.sum(-1).sum(-1)  # Regularization for white background
    return c + 1 - weight_sum.unsqueeze(-1)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

#Neural Radiance Field (NeRF) model
class NeRF(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NeRF, self).__init__()
        self.pos_encoding = nn.Linear(input_dim, hidden_dim)
        self.view_dep_encoding = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, pos, view_dir):
        pos_encoded = self.pos_encoding(pos)
        view_dep_input = torch.cat([pos_encoded, view_dir], dim=1)
        view_dep_output = self.view_dep_encoding(view_dep_input)
        return view_dep_output






